# M1 Notebook 13 — Conditional Probability and Bayesian Updating

**Notebook ID:** M1_N13  
**Status:** Runnable first edition  
**Random seed:** 42

> Bayesian updating revises uncertainty when new evidence arrives.


## 1. Learning objectives

1. Compute conditional probability.
2. Apply the law of total probability.
3. Derive and use Bayes' theorem.
4. Interpret prior, likelihood, evidence, and posterior.
5. Use odds and likelihood ratios.
6. Perform sequential Bayesian updating.
7. Apply Bayesian reasoning to diagnosis, forecasting, and Decision Intelligence.


In [ ]:
from srai_math.utils import environment_info, set_seed
from srai_math.probability import (
    bayes_binary,
    bayes_discrete,
    odds_to_probability,
    posterior_odds,
    probability_to_odds,
    sequential_bayes,
    total_probability,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
set_seed(42)
environment_info()


## 2. Conditional probability

For events \(A\) and \(B\),

\[
P(A\mid B)
=
\frac{P(A\cap B)}{P(B)},
\qquad P(B)>0.
\]


## 3. Law of total probability

If \(H_1,\ldots,H_k\) form a partition, then

\[
P(E)
=
\sum_{i=1}^{k}
P(E\mid H_i)P(H_i).
\]


In [ ]:
priors = [0.4, 0.6]
likelihoods = [0.2, 0.5]

evidence_probability = total_probability(
    priors,
    likelihoods,
)

assert np.isclose(evidence_probability, 0.38)
evidence_probability


## 4. Bayes' theorem

\[
P(H_i\mid E)
=
\frac{P(E\mid H_i)P(H_i)}
{\sum_j P(E\mid H_j)P(H_j)}.
\]


In [ ]:
posterior = bayes_discrete(
    priors={"Model_A": 0.5, "Model_B": 0.5},
    likelihoods={"Model_A": 0.8, "Model_B": 0.2},
)

posterior


## 5. Medical-test example

Suppose:

- prevalence \(=1\%\);
- sensitivity \(=95\%\);
- specificity \(=90\%\).

What is the probability of the condition after a positive test?


In [ ]:
positive_posterior = bayes_binary(
    prior=0.01,
    sensitivity=0.95,
    specificity=0.90,
    positive=True,
)

positive_posterior


A positive result raises the probability substantially, but the posterior remains below 10% because the condition is rare and false positives are not negligible.


## 6. Base-rate comparison

In [ ]:
prevalences = np.array([0.001, 0.01, 0.05, 0.10, 0.25])
posteriors = [
    bayes_binary(
        prior=p,
        sensitivity=0.95,
        specificity=0.90,
        positive=True,
    )
    for p in prevalences
]

pd.DataFrame({
    "prior_prevalence": prevalences,
    "posterior_after_positive": posteriors,
})


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(prevalences, posteriors, marker="o")
ax.set_xlabel("Prior probability")
ax.set_ylabel("Posterior after positive evidence")
ax.set_title("Base Rates Matter in Bayesian Updating")
plt.show()


## 7. Negative evidence

For a negative result,

\[
P(H\mid E^-)
=
\frac{P(E^-\mid H)P(H)}
{P(E^-\mid H)P(H)+P(E^-\mid H^c)P(H^c)}.
\]


In [ ]:
negative_posterior = bayes_binary(
    prior=0.10,
    sensitivity=0.90,
    specificity=0.80,
    positive=False,
)

negative_posterior


## 8. Odds form

Bayes' theorem can be written as

\[
\text{Posterior odds}
=
\text{Prior odds}
\times
\text{Likelihood ratio}.
\]


In [ ]:
prior_probability = 0.20
prior_odds = probability_to_odds(prior_probability)
likelihood_ratio = 3.0
updated_odds = posterior_odds(prior_odds, likelihood_ratio)
updated_probability = odds_to_probability(updated_odds)

{
    "prior_odds": prior_odds,
    "posterior_odds": updated_odds,
    "posterior_probability": updated_probability,
}


## 9. Sequential Bayesian updating

Independent evidence items can be incorporated one after another.


In [ ]:
history = sequential_bayes(
    prior={
        "Drought": 0.30,
        "Normal": 0.50,
        "Wet": 0.20,
    },
    evidence_likelihoods=[
        {
            "Drought": 0.70,
            "Normal": 0.25,
            "Wet": 0.05,
        },
        {
            "Drought": 0.60,
            "Normal": 0.30,
            "Wet": 0.10,
        },
    ],
)

pd.DataFrame(history, index=["Prior", "After evidence 1", "After evidence 2"])


## 10. Visualizing sequential updates

In [ ]:
history_frame = pd.DataFrame(
    history,
    index=["Prior", "Evidence 1", "Evidence 2"],
)

fig, ax = plt.subplots(figsize=(8, 4))
history_frame.plot(kind="bar", ax=ax)
ax.set_ylabel("Probability")
ax.set_title("Sequential Bayesian Updating")
ax.legend(title="Hypothesis")
plt.xticks(rotation=0)
plt.show()


## 11. Sensitivity to test specificity

In [ ]:
specificities = np.linspace(0.70, 0.999, 100)
posterior_curve = [
    bayes_binary(
        prior=0.01,
        sensitivity=0.95,
        specificity=s,
        positive=True,
    )
    for s in specificities
]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(specificities, posterior_curve)
ax.set_xlabel("Specificity")
ax.set_ylabel("Posterior probability")
ax.set_title("Posterior Sensitivity to Specificity")
plt.show()


For rare conditions, small changes in specificity may materially affect the posterior because false positives can dominate true positives.


## 12. Statistics interpretation

Bayesian inference combines prior information with data likelihoods. The posterior becomes the basis for estimation, prediction, and future updating.


## 13. AI interpretation

Bayesian reasoning supports:

- probabilistic classification;
- Bayesian neural networks;
- uncertainty calibration;
- sensor fusion;
- probabilistic graphical models;
- sequential learning;
- active learning.


## 14. Decision Intelligence case — Updating drought risk

A government begins with a prior drought probability. Satellite vegetation stress and seasonal-forecast evidence arrive sequentially.


In [ ]:
drought_history = sequential_bayes(
    prior={
        "Drought": 0.25,
        "No_Drought": 0.75,
    },
    evidence_likelihoods=[
        {
            "Drought": 0.80,
            "No_Drought": 0.20,
        },
        {
            "Drought": 0.70,
            "No_Drought": 0.35,
        },
        {
            "Drought": 0.90,
            "No_Drought": 0.25,
        },
    ],
)

drought_frame = pd.DataFrame(
    drought_history,
    index=[
        "Prior",
        "Satellite evidence",
        "Seasonal forecast",
        "Field reports",
    ],
)

drought_frame


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(
    drought_frame.index,
    drought_frame["Drought"],
    marker="o",
)
ax.set_ylim(0, 1)
ax.set_ylabel("Posterior drought probability")
ax.set_title("Evidence-Driven Drought Risk Updating")
plt.xticks(rotation=20)
plt.show()


### Interpretation

Bayesian updating makes the evidence pathway explicit. Decision thresholds, costs, uncertainty about likelihoods, dependence among evidence sources, and ethical implications remain separate governance questions.


## 15. Engineering notes

- Posterior results depend on prior and likelihood quality.
- Sequential updates must account for dependence among evidence sources.
- Extremely small probabilities are often handled in log space.
- Bayesian models should report sensitivity to alternative priors.
- Calibration matters more than apparent numerical precision.
- Posterior probability is not the same as a recommended action.


## 16. Common errors

- Ignoring base rates.
- Confusing \(P(E\mid H)\) with \(P(H\mid E)\).
- Treating evidence sources as independent without justification.
- Using arbitrary likelihoods.
- Reporting posterior probabilities without uncertainty or sensitivity analysis.
- Converting a posterior directly into policy without a decision model.


## 17. Exercises

### Level A
Identify prior, likelihood, evidence, and posterior.

### Level B
Derive Bayes' theorem from conditional probability.

### Level C
Implement a log-space discrete Bayesian updater.

### Capstone
Build a sequential risk-updating model, document each likelihood assumption, test prior sensitivity, and define a separate decision rule for action.


## 18. Key insight

Bayesian updating transforms evidence into revised uncertainty. It does not eliminate uncertainty and does not by itself choose an action. It provides a disciplined bridge from prior beliefs to posterior knowledge.
